<a target="_blank" href="https://colab.research.google.com/">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# A Subthreshold Variable-Gain Amplifier Core for Hearing-Aid-Class Audio Compression: gm/ID-Driven Automated Sizing and Layout in SKY130

```
Submission to the IEEE SSCS Open-Source Ecosystem "Code-a-Chip" Travel Grant Awards at ISSCC'27
```

| Name | Affiliation | Lab / Group | IEEE / SSCS Member | Email |
|:--|:--|:--|:--:|:--|
| **Ayush \<Surname\>** *(travel-grant recipient)* | VIT Vellore, India | \<Lab\> | \<Yes/No\> | \<email\> |
| \<Team member 2\> | VIT Vellore, India | \<Lab\> | \<Yes/No\> | \<email\> |
| \<Faculty advisor\> | VIT Vellore, India | \<Lab\> | \<Yes/No\> | \<email\> |

**Work licensed under Apache License 2.0** — `SPDX-License-Identifier: Apache-2.0`.
Full terms in the `LICENSE` file at the top level of this submission directory.

---

### Reuse and attribution

This notebook's methodology builds on prior Code-a-Chip submissions and open-source projects. All reused
material is credited here and in Section 12.

| Reused artefact | Source | How it is used here |
|:--|:--|:--|
| gm/ID (C/ID) LUT-driven sizing flow | A. Adair, *Code-a-Chip* ISSCC'25 | Device characterisation and inverse sizing methodology |
| LUT + `pymoo` optimisation structure | A. M. Elhelly, *Code-a-Chip* ISSCC'26 | Multi-objective optimiser wrapper pattern |
| \<any other notebook / repo you reuse\> | \<link\> | \<what you took\> |

---

### TL;DR

We present a **reproducible, LUT-driven automated design flow** for a subthreshold variable-gain
amplifier (VGA) core intended as the gain element of an ultra-low-power hearing-aid audio compressor,
implemented entirely with open-source tools on the SkyWater SKY130 PDK. The flow characterises the
device once into a gm/ID lookup table, solves the sizing problem *inversely* from a target
specification, sweeps the achievable design space, and emits a DRC/LVS-clean layout together with a
pre- versus post-layout figure-of-merit comparison. **The contribution is the flow, not a single
sized circuit**: any subthreshold transconductor with the same topology can be re-synthesised from a
new specification by editing one dictionary.

> **Scope statement (read this).** This notebook delivers the **variable-gain core** of a dynamic-range
> compressor. The envelope detector and the closed gain-control loop are **not** implemented here; the
> compression characteristic (threshold, ratio, attack/release) is therefore **not** claimed. See
> Section 10.2.


---
## 0. How to read this notebook

### 0.1 Notation

Symbols follow the conventions below throughout. Lower-case italic with lower-case subscript denotes a
small-signal quantity; upper-case denotes a large-signal or bias quantity.

| Symbol | Meaning | Unit |
|:--:|:--|:--:|
| $I_D$ | Drain bias current | A |
| $g_m$ | Small-signal transconductance | S |
| $g_{ds}$ | Small-signal output conductance | S |
| $g_m/I_D$ | Transconductance efficiency (the sizing coordinate) | V$^{-1}$ |
| $I_D/W$ | Current density (the second sizing coordinate) | A/m |
| $V_{ov}=V_{GS}-V_{TH}$ | Overdrive voltage | V |
| $n$ | Subthreshold slope factor | – |
| $U_T = kT/q$ | Thermal voltage ($\approx 25.9$ mV at 300 K) | V |
| $I_C$ | Inversion coefficient | – |
| $A_v$ | Small-signal voltage gain | V/V (dB) |
| $f_{-3\mathrm{dB}}$, $f_u$ | $-3$ dB and unity-gain frequency | Hz |
| $\mathrm{PM}$ | Phase margin | ° |
| $\overline{v_{n,\mathrm{in}}^2}$ | Input-referred noise PSD | V$^2$/Hz |
| $\mathrm{THD}$ | Total harmonic distortion | % (dB) |
| $\mathrm{NEF}$ | Noise efficiency factor | – |
| $V_{\mathrm{ctrl}}$ | Gain-control voltage | V |

Design-space coordinates are written as the ordered pair $\left(g_m/I_D,\; I_D/W\right)$.
PVT corners are written `<model>/<T>/<VDD>`, e.g. `ss/85C/1.62V`.

### 0.2 Cell conventions

Every code cell carries one tag in its first comment line so that a reader can run a subset of the
notebook without executing multi-hour steps:

| Tag | Meaning | Typical runtime |
|:--|:--|:--|
| `#@ SETUP` | Installs / environment. Run once. | minutes |
| `#@ CHAR` | Device characterisation sweep (expensive). Cached artefacts provided. | ~1 h |
| `#@ DESIGN` | Sizing / optimisation from cached LUTs. Cheap. | seconds |
| `#@ VERIFY` | SPICE verification of a sized instance. | minutes |
| `#@ LAYOUT` | Layout generation and DRC/LVS. | minutes |
| `#@ FIG` | Produces a figure or table in the results section. | seconds |

**Default run mode is `REBUILD_LUT = False`** — cached characterisation data ships with the submission
so a reviewer can reproduce every result in Sections 5–9 in under ten minutes.

### 0.3 Figure and table conventions

Figures are captioned below the artefact as *Figure N. Caption.* and tables above as *Table N. Caption.*
Every figure that reports a measured quantity states the corner and the testbench it came from.


---
## 1. Introduction

### 1.1 Background

Hearing aids compress the wide dynamic range of ambient acoustic signals into the narrow residual
dynamic range of an impaired ear. The gain element that performs this compression sits directly in the
signal path, so its noise floor sets the noise floor of the whole instrument, and its quiescent current
dominates the battery budget of a device that must run for days from a zinc-air cell of a few tens of
mAh.

\<Expand: 2–4 paragraphs. State the power and noise budget of a hearing-aid front end with numbers, cite
the literature you are targeting, and explain why subthreshold operation is the natural regime — the
$g_m/I_D$ of a MOSFET is maximised and approximately $1/(nU_T)$ in weak inversion, which is exactly the
figure a noise- and power-constrained gain stage wants to maximise.\>

### 1.2 Motivation

Commercial hearing-aid ICs are closed. \<Expand: state the access/cost gap that motivates an open
implementation, and state what a designer today cannot do — namely, re-derive a gain stage against
their own specification without a proprietary flow.\>

The specific gap this notebook addresses is a **methodological** one. Sizing a subthreshold
transconductor by hand is an iterative, unrecorded process: a designer guesses $W/L$, simulates, and
adjusts. That process is neither reproducible nor transferable. A $g_m/I_D$-based flow inverts it — the
designer states the specification, and the sizing is *solved* from pre-characterised device data.

### 1.3 Contributions

1. An open, re-runnable **device characterisation** of the SKY130 devices in weak and moderate inversion,
   emitted as a lookup table over $\left(g_m/I_D,\; I_D/W\right)$.
2. An **inverse sizing procedure** that maps a VGA specification (gain, bandwidth, noise, power) onto
   device operating points without manual iteration.
3. A **multi-objective sweep** of the achievable design space, exposing the noise–power–gain trade-off
   as a Pareto front rather than a single point.
4. A **verified physical implementation** of one selected operating point: DRC- and LVS-clean layout with
   pre- versus post-layout FoM comparison.
5. A packaged, cached, one-click-reproducible artefact: every number and figure in Sections 5–9 is
   regenerated by running this notebook top to bottom.

### 1.4 Notebook overview

\<Insert `images/flow.png` here — a single block diagram of the whole flow. Every accepted analog
submission surveyed opens with one.\>

The flow proceeds as:

1. **§2** Environment, tool versions, PDK, and run modes.
2. **§3** Circuit topology, governing equations, and the target specification datasheet.
3. **§4** Device characterisation into a $g_m/I_D$ lookup table.
4. **§5** Inverse sizing: specification $\rightarrow$ operating point $\rightarrow$ device geometry.
5. **§6** Design-space exploration and multi-objective optimisation.
6. **§7** SPICE verification and figure-of-merit extraction across PVT.
7. **§8** Layout generation, DRC, LVS, parasitic extraction.
8. **§9** Results: pre/post-layout FoM table and comparison against prior art.
9. **§10** Discussion and limitations; **§11** conclusion; **§12** references.


---
## 2. Environment configuration and reproducibility

### 2.1 Tool versions

*Table 1. Tool versions this notebook was developed and validated against.* Pin these; a reviewer who
installs "latest" a year from now will not reproduce your numbers otherwise.

| Tool | Version | Purpose |
|:--|:--|:--|
| ngspice | \<e.g. 42\> | Transistor-level simulation |
| xschem | \<e.g. 3.4.6\> | Schematic capture / netlist export |
| Magic | \<e.g. 8.3.x\> | Layout editing, DRC, parasitic extraction |
| Netgen | \<e.g. 1.5.x\> | LVS |
| KLayout | \<e.g. 0.29.x\> | Layout viewing and figure export |
| open_pdks / sky130A | \<version\> | PDK |
| Python | \<3.11\> | Flow orchestration |
| numpy / scipy / pandas / matplotlib | \<versions\> | Data handling and plotting |
| pymoo | \<0.6.x\> | Multi-objective optimisation |

### 2.2 Run modes

Set these flags once; everything downstream honours them.


In [ ]:
#@ SETUP
# ---------------------------------------------------------------------------
# Run-mode flags. Defaults are chosen so a reviewer reproduces every result
# in Sections 5-9 in minutes, using the cached artefacts shipped with this
# submission. Set to True only if you want to regenerate from scratch.
# ---------------------------------------------------------------------------
REBUILD_LUT   = False   # re-run the device characterisation sweep (~1 h)
RERUN_OPT     = False   # re-run the multi-objective optimisation (~10 min)
RERUN_SPICE   = True    # re-run verification of the selected design point
RERUN_LAYOUT  = False   # regenerate layout + DRC/LVS (needs local Magic/Netgen)

SEED = 42               # fixed for every stochastic step, so results are exact-reproducible

import os, sys, json, subprocess, textwrap
from pathlib import Path

ROOT   = Path.cwd()
CACHE  = ROOT / "cache"      # shipped: LUTs, optimiser results, extracted netlists
FIGS   = ROOT / "images"
NETL   = ROOT / "netlists"
for d in (CACHE, FIGS, NETL):
    d.mkdir(exist_ok=True)

print(f"root   : {ROOT}")
print(f"cache  : {CACHE}  (exists: {CACHE.exists()})")


In [ ]:
#@ SETUP
# ---------------------------------------------------------------------------
# Tool installation. Two paths are supported and BOTH are documented, because
# reviewers run notebooks in different places:
#   (a) Colab / bare cloud VM  -> micromamba, packages from the litex-hub channel
#   (b) Local IIC-OSIC-TOOLS container -> tools already on $PATH, skip install
# ---------------------------------------------------------------------------
def have(tool):
    from shutil import which
    return which(tool) is not None

REQUIRED = ["ngspice", "magic", "netgen", "klayout"]
missing  = [t for t in REQUIRED if not have(t)]

if not missing:
    print("All tools already on $PATH (IIC-OSIC-TOOLS container detected). Skipping install.")
else:
    print(f"Missing: {missing}. Installing via micromamba ...")
    # <Insert your micromamba / apt install block here. Pin exact versions.>
    # Keep this cell idempotent: re-running it on a provisioned machine must be a no-op.


In [ ]:
#@ SETUP
# ---------------------------------------------------------------------------
# Record the environment actually used, and emit it into the submission so the
# reported numbers are traceable to a concrete toolchain.
# ---------------------------------------------------------------------------
import platform
env = {"python": sys.version.split()[0], "platform": platform.platform()}
for t in REQUIRED:
    try:
        out = subprocess.run([t, "--version"], capture_output=True, text=True, timeout=20)
        env[t] = (out.stdout or out.stderr).strip().splitlines()[0]
    except Exception as e:
        env[t] = f"<unavailable: {e}>"
env["PDK_ROOT"] = os.environ.get("PDK_ROOT", "<unset>")

(CACHE / "environment.json").write_text(json.dumps(env, indent=2))
for k, v in env.items():
    print(f"{k:12s}: {v}")


---
## 3. Topology, governing equations, and target specification

### 3.1 Why a subthreshold variable-$g_m$ transconductor

\<Insert `images/schematic_vga.png` — the annotated schematic, with every device labelled to match the
symbols used below.\>
*Figure 2. VGA core schematic. Device labels correspond to Table 3.*

In weak inversion the drain current of an MOS device is exponential in gate–source voltage,

$$
I_D \;=\; I_{S}\,\frac{W}{L}\,\exp\!\left(\frac{V_{GS}-V_{TH}}{n U_T}\right)
\left[1-\exp\!\left(-\frac{V_{DS}}{U_T}\right)\right],
$$

so the transconductance is proportional to the bias current,

$$
g_m \;=\; \frac{\partial I_D}{\partial V_{GS}} \;=\; \frac{I_D}{n U_T},
\qquad\Longrightarrow\qquad
\frac{g_m}{I_D}\Big|_{\text{weak inv.}} \;=\; \frac{1}{n U_T} \;\approx\; \text{const}.
$$

This is the single fact the whole design rests on: **$g_m$ is set by current alone**, which makes gain
electronically controllable by a bias current with no change of geometry. The small-signal gain of the
stage is

$$
A_v \;=\; \frac{g_m}{g_{ds,\mathrm{tot}}}
\;=\; \frac{g_m/I_D}{\left(g_{ds}/I_D\right)_{\!\mathrm{tot}}},
$$

written deliberately in *current-normalised* form because that is the form the lookup table stores, and
it makes $A_v$ independent of the absolute current — the property that lets us size the device once and
scale the bias afterwards.

The dominant pole and unity-gain frequency follow as

$$
f_{-3\mathrm{dB}} = \frac{g_{ds,\mathrm{tot}}}{2\pi C_L},
\qquad
f_u = \frac{g_m}{2\pi C_L} = \frac{1}{2\pi C_L}\left(\frac{g_m}{I_D}\right) I_D .
$$

Input-referred thermal noise of the input pair, referred through the stage,

$$
\overline{v_{n,\mathrm{in}}^{2}} \;=\; \frac{4kT\,\gamma\,n}{g_m}
\;+\; \underbrace{\frac{K_f}{C_{ox} W L\, f}}_{\text{flicker}},
$$

exposes the fundamental tension this design must resolve: noise falls as $1/g_m$ and therefore as
$1/I_D$, while power rises linearly with $I_D$. The noise efficiency factor collapses that trade-off
into one number,

$$
\mathrm{NEF} \;=\; V_{n,\mathrm{rms}} \sqrt{\frac{2 I_{\mathrm{tot}}}{\pi U_T \cdot 4kT \cdot \mathrm{BW}}},
$$

and is the headline FoM reported in Section 9.

### 3.2 Gain-control law

\<State the gain-control mechanism explicitly and derive $A_v(V_{\mathrm{ctrl}})$. If gain is controlled
by steering the tail current, show $A_v \propto \ldots$; give the intended control range in dB and the
control voltage range that spans it. This equation is what Section 7.2 measures against.\>

$$
A_v\!\left(V_{\mathrm{ctrl}}\right) \;=\; \langle\text{your derived expression}\rangle
$$

### 3.3 Target specification

*Table 2. Target specification of the VGA core.* Targets are stated **before** any simulation; Section 9
reports achieved values against this table. Do not retrofit this table to your results.

| Parameter | Symbol | Target | Unit | Rationale / source |
|:--|:--:|:--:|:--:|:--|
| Supply voltage | $V_{DD}$ | \<1.8\> | V | SKY130 nominal |
| Total quiescent current | $I_{\mathrm{tot}}$ | \< \<x\> | µA | Hearing-aid battery budget \<cite\> |
| Mid-band gain range | $A_v$ | \<x\> to \<y\> | dB | Compression range required \<cite\> |
| $-3$ dB bandwidth | $f_{-3\mathrm{dB}}$ | $\geq$ \<10\> | kHz | Audio band |
| Phase margin | PM | $\geq 60$ | ° | Stability under $C_L$ |
| Input-referred noise | $v_{n,\mathrm{in}}$ | $\leq$ \<x\> | nV/$\sqrt{\mathrm{Hz}}$ | Below acoustic noise floor \<cite\> |
| Integrated input noise | $V_{n,\mathrm{rms}}$ | $\leq$ \<x\> | µV$_{\mathrm{rms}}$ | 20 Hz – 20 kHz |
| THD @ \<x\> mV$_{pp}$ | THD | $\leq$ \<1\> | % | Audio quality |
| Noise efficiency factor | NEF | $\leq$ \<x\> | – | Comparison against prior art |
| Load capacitance | $C_L$ | \<x\> | pF | Next-stage input |
| Core area | $A$ | $\leq$ \<x\> | µm² | – |

*Table 3. Device inventory.* Fill after sizing; keep the symbol names identical to Figure 2.

| Device | Role | $W/L$ | $I_D$ | $g_m/I_D$ | Inversion region |
|:--|:--|:--:|:--:|:--:|:--:|
| M1, M2 | Input pair | | | | weak |
| M3, M4 | Load | | | | |
| M5 | Tail / gain control | | | | |


---
## 4. Device characterisation: the $g_m/I_D$ lookup table

This is the step that makes the flow reusable. We sweep the device **once** over bias and geometry, and
store the small-signal parameters normalised to $I_D$. Every later sizing decision is then a table lookup
plus interpolation, not a simulation.

### 4.1 Characterisation testbench

\<Insert `images/char_tb.png` — the characterisation testbench schematic.\>
*Figure 3. Device characterisation testbench. $V_{GS}$ and $V_{DS}$ are swept at fixed $L$; the sweep is
repeated for each $L$ in the device table.*

The sweep records, for each $\left(L,\,V_{GS},\,V_{DS}\right)$ point: $I_D/W$, $g_m/I_D$, $g_{ds}/I_D$,
$C_{gs}/I_D$, $C_{gd}/I_D$, and the saturation margin $V_{DS}-V_{DS,\mathrm{sat}}$. Storing everything
normalised to $I_D$ is what makes the table geometry-independent.


In [ ]:
#@ CHAR
# ---------------------------------------------------------------------------
# Generate the gm/ID lookup table. EXPENSIVE. Skipped by default: the cached
# table shipped in cache/lut_sky130.parquet reproduces every downstream result.
# ---------------------------------------------------------------------------
LUT_PATH = CACHE / "lut_sky130.parquet"

if REBUILD_LUT:
    # <Emit the ngspice .dc sweep from the template in netlists/char_tb.cir,
    #  parse the raw output, and assemble the normalised table.>
    # Sweep grid (state it explicitly - a reviewer must be able to see the resolution):
    L_LIST   = [0.15, 0.35, 0.5, 1.0, 2.0]        # um
    VGS_GRID = (0.0, 1.2, 0.005)                   # start, stop, step  [V]
    VDS_GRID = (0.0, 1.8, 0.05)                    # start, stop, step  [V]
    raise NotImplementedError("<characterisation sweep>")
else:
    print(f"Using cached LUT: {LUT_PATH}")


In [ ]:
#@ CHAR #@ FIG
# ---------------------------------------------------------------------------
# Sanity-check the LUT before trusting anything built on it.
# The two canonical plots below are the ones a reviewer looks for.
# ---------------------------------------------------------------------------
# Figure 4a: gm/ID versus ID/W, one curve per L  -> the sizing chart itself
# Figure 4b: gm/ID versus inversion coefficient  -> shows weak/moderate/strong
#            boundaries and confirms the intended operating region is covered
# <plotting code>


*Figure 4. (a) $g_m/I_D$ versus current density $I_D/W$ for each channel length; (b) $g_m/I_D$ versus
inversion coefficient, with the weak / moderate / strong inversion boundaries marked. The shaded band is
the region this design operates in.*

\<One paragraph reading the chart: state the maximum achievable $g_m/I_D$ you measure and compare it to
the theoretical weak-inversion asymptote $1/(nU_T)$; extract $n$ from the fit and report it. This
sentence proves the LUT is physically sane and is worth more to a reviewer than the plot itself.\>


---
## 5. Inverse sizing: specification $\rightarrow$ geometry

Given the target of Table 2, sizing is now a solve rather than a search. For a chosen $g_m/I_D$:

$$
I_D = \frac{2\pi C_L f_u}{\left(g_m/I_D\right)},
\qquad
W = \frac{I_D}{\left(I_D/W\right)\big|_{g_m/I_D,\,L}},
\qquad
A_v = \frac{g_m/I_D}{\left(g_{ds}/I_D\right)}.
$$

The only free choices left are $g_m/I_D$ and $L$ — a two-dimensional space that Section 6 sweeps
exhaustively.


In [ ]:
#@ DESIGN
# ---------------------------------------------------------------------------
# Interpolators over the LUT. scipy RegularGridInterpolator on the
# (L, gm_over_id) grid, returning ID/W, gds/ID, cgs/ID, cgd/ID.
# ---------------------------------------------------------------------------
# <build interpolators>

def size_from_spec(spec, gm_id, L):
    """Solve device geometry from a target specification.

    Parameters
    ----------
    spec  : dict  target specification (fu_Hz, CL_F, Av_min, ...)
    gm_id : float chosen transconductance efficiency [1/V]
    L     : float channel length [um]

    Returns
    -------
    dict with W_um, ID_A, Av_dB, fu_Hz, vn_in, P_W
    """
    # <implement: the three equations above, then evaluate the resulting FoM>
    raise NotImplementedError


---
## 6. Design-space exploration and multi-objective optimisation

A single sized instance is a data point; the **trade-off surface** is the result. We minimise
$\left(P,\; V_{n,\mathrm{rms}}\right)$ subject to gain, bandwidth, phase-margin and area constraints, over
the decision variables $\left(g_m/I_D,\, L,\, \ldots\right)$, using NSGA-II.

$$
\begin{aligned}
\min_{\mathbf{x}} \quad & \mathbf{f}(\mathbf{x}) = \left[\; P(\mathbf{x}),\;\; V_{n,\mathrm{rms}}(\mathbf{x}) \;\right] \\
\text{s.t.} \quad & A_v(\mathbf{x}) \ \geq\ A_{v,\min}, \\
                  & f_{-3\mathrm{dB}}(\mathbf{x}) \ \geq\ f_{\min}, \\
                  & \mathrm{PM}(\mathbf{x}) \ \geq\ 60^\circ, \\
                  & \mathbf{x} \in \left[\mathbf{x}_{\mathrm{lb}},\, \mathbf{x}_{\mathrm{ub}}\right].
\end{aligned}
$$


In [ ]:
#@ DESIGN
# ---------------------------------------------------------------------------
# pymoo problem definition. Objectives evaluated from the LUT interpolators,
# NOT from SPICE - that is what makes the sweep tractable. The selected point
# is then confirmed in SPICE in Section 7.
# ---------------------------------------------------------------------------
# from pymoo.core.problem import ElementwiseProblem
# from pymoo.algorithms.moo.nsga2 import NSGA2
# from pymoo.optimize import minimize
# <define VGAProblem, run NSGA2 with seed=SEED, cache result to CACHE>


In [ ]:
#@ FIG
# ---------------------------------------------------------------------------
# Figure 5: Pareto front, power vs integrated input-referred noise, with the
# selected design point marked and the specification limits drawn as lines.
# ---------------------------------------------------------------------------
# <plotting code>


*Figure 5. Pareto front of quiescent power against integrated input-referred noise. Dashed lines are
the Table 2 limits; the marked point is the design carried forward to layout.*

\<Two or three sentences of engineering reading: where is the knee, what does moving along the front cost
in each direction, and **why** you selected the point you selected. A reviewer rewards the reasoning far
more than the plot.\>

### 6.1 Selected design point

*Table 4. Selected operating point and resulting geometry.*

| Quantity | Value | Unit |
|:--|:--:|:--:|
| $g_m/I_D$ | | V$^{-1}$ |
| $L$ | | µm |
| $W$ | | µm |
| $I_D$ (per branch) | | µA |
| Predicted $A_v$ | | dB |
| Predicted $f_{-3\mathrm{dB}}$ | | kHz |
| Predicted $V_{n,\mathrm{rms}}$ | | µV$_{\mathrm{rms}}$ |


---
## 7. SPICE verification and figure-of-merit extraction

The LUT predicts; SPICE confirms. Every quantity in Table 2 is now measured on the sized netlist. **The
prediction-versus-simulation comparison is itself a result** — it validates the flow, and Section 9
reports the residual error.

### 7.1 Operating point and DC


In [ ]:
#@ VERIFY
# ---------------------------------------------------------------------------
# .op  -> confirm every device is in the intended inversion region.
# Assert it in code rather than eyeballing it: a reviewer sees the check.
# ---------------------------------------------------------------------------
# <run ngspice .op, parse, tabulate device region, assert weak inversion for M1/M2>


### 7.2 Gain control characteristic — $A_v$ versus $V_{\mathrm{ctrl}}$

This is the defining measurement of a VGA and the one that connects the block to its role in a
compressor. Overlay the analytic law from Section 3.2 on the simulated points.


In [ ]:
#@ VERIFY #@ FIG
# Figure 6: Av (dB) vs Vctrl, simulated vs analytic model of Section 3.2.
# Report: control range in dB, control-law linearity (dB error vs model),
# and the monotonic range.


### 7.3 AC response: gain, bandwidth, phase margin

In [ ]:
#@ VERIFY #@ FIG
# Figure 7: Bode magnitude and phase, one trace per gain setting.
# Extract and tabulate: Av_dB, f_-3dB, f_u, PM for each setting.


### 7.4 Noise

In [ ]:
#@ VERIFY #@ FIG
# Figure 8: input-referred noise PSD; mark the 1/f corner.
# Integrate 20 Hz - 20 kHz -> Vn_rms. Compute NEF. Report both.


### 7.5 Linearity

In [ ]:
#@ VERIFY #@ FIG
# Figure 9: THD vs input amplitude at each gain setting (transient + FFT).
# Report THD at the Table 2 test amplitude and the 1% THD input level.


### 7.6 PVT corners

Subthreshold circuits are exponentially sensitive to $V_{TH}$ and temperature; a subthreshold design
reported only at `tt/27C` is not a characterised design. Sweep the full corner set.

*Table 5. FoM across PVT.* Corners: {`tt`,`ss`,`ff`,`sf`,`fs`} $\times$ {$-20$, 27, 60, 85} °C
$\times$ {$0.9V_{DD}$, $V_{DD}$, $1.1V_{DD}$}.

| Corner | $A_v$ (dB) | $f_{-3\mathrm{dB}}$ (kHz) | PM (°) | $I_{\mathrm{tot}}$ (µA) | $V_{n,\mathrm{rms}}$ (µV) | THD (%) |
|:--|:--:|:--:|:--:|:--:|:--:|:--:|
| `tt/27C/1.80V` | | | | | | |
| `ss/85C/1.62V` | | | | | | |
| `ff/-20C/1.98V` | | | | | | |
| … | | | | | | |


In [ ]:
#@ VERIFY #@ FIG
# Figure 10: FoM spread across corners (e.g. box plot or spider chart).
# State the worst-case corner explicitly and whether the spec still holds there.


### 7.7 Monte Carlo mismatch

\<If your gain accuracy matters — and for a compressor it does — run N=\<200\> Monte Carlo instances and
report the $\sigma$ of gain and of input-referred offset. This is exactly the axis the ISSCC'26 EEG
instrumentation-amplifier submission built its entire contribution on, and it is cheap to add once §7.3
works.\>


In [ ]:
#@ VERIFY #@ FIG
# Figure 11: histogram of Av and input-referred offset over Monte Carlo runs.
# Report mean, sigma, and 3-sigma spread.


---
## 8. Physical implementation

### 8.1 Layout strategy

\<State the matching strategy before showing the picture: common-centroid or interdigitated input pair,
dummy devices, guard ring, symmetric routing of the differential nets. Explain *why* each choice follows
from the mismatch requirement in Section 7.7 — this paragraph is what distinguishes a designed layout
from a drawn one.\>

\<Insert `images/layout.png` (KLayout render) and `images/layout_annotated.png` (with blocks labelled).\>
*Figure 12. Core layout. Area = \<x\> µm × \<y\> µm = \<z\> µm².*


In [ ]:
#@ LAYOUT
# ---------------------------------------------------------------------------
# DRC (Magic) and LVS (Netgen). Print the actual reports - do not paraphrase
# them. "DRC clean" without the report is an unverifiable claim.
# ---------------------------------------------------------------------------
# <run magic drc, print count; run netgen lvs, print the final match line>


### 8.2 Post-layout verification

*Table 6. Pre- versus post-layout figure-of-merit comparison.* **This table is the payoff of doing
layout at all.** A layout without an extracted re-simulation demonstrates drawing; this demonstrates
verification.

| FoM | Schematic | Post-layout (RC-extracted) | Δ | Spec | Pass |
|:--|:--:|:--:|:--:|:--:|:--:|
| $A_v$ (dB) | | | | | |
| $f_{-3\mathrm{dB}}$ (kHz) | | | | | |
| PM (°) | | | | | |
| $I_{\mathrm{tot}}$ (µA) | | | | | |
| $V_{n,\mathrm{rms}}$ (µV$_{\mathrm{rms}}$) | | | | | |
| THD (%) | | | | | |
| NEF | | | | | |

\<One paragraph explaining the deltas physically — which parasitic caused which shift. "Bandwidth fell
12% due to the extracted 40 fF on the output node" is a result; "post-layout results are slightly worse"
is not.\>


---
## 9. Results

### 9.1 Specification compliance

*Table 7. Achieved performance against the Table 2 targets.* Report honestly, including any target
missed — a missed target with a diagnosed cause reads as engineering; a table with no misses reads as a
table that was written after the fact.

| Parameter | Target | Achieved (post-layout, `tt/27C`) | Worst corner | Met? |
|:--|:--:|:--:|:--:|:--:|
| … | | | | |

### 9.2 Flow validation: predicted versus simulated

*Table 8. LUT prediction error.* This quantifies the central claim of the notebook — that the flow sizes
correctly without iteration.

| Quantity | LUT prediction | SPICE | Relative error |
|:--|:--:|:--:|:--:|
| $A_v$ | | | |
| $f_u$ | | | |
| $V_{n,\mathrm{rms}}$ | | | |

### 9.3 Comparison with prior art

*Table 9. Comparison against published hearing-aid / biomedical gain stages.* Include the process,
supply, current, noise, NEF and whether the work is open. Being **open** is a column you win on; use it.

| | This work | \<Ref A\> | \<Ref B\> | \<Ref C\> |
|:--|:--:|:--:|:--:|:--:|
| Process | SKY130 (open) | | | |
| Supply (V) | | | | |
| $I_{\mathrm{tot}}$ (µA) | | | | |
| Gain range (dB) | | | | |
| $v_{n,\mathrm{in}}$ (nV/$\sqrt{\mathrm{Hz}}$) | | | | |
| NEF | | | | |
| Area (µm²) | | | | |
| Open-source | **Yes** | No | No | No |


---
## 10. Discussion

### 10.1 What the flow does and does not generalise to

\<Two or three numbered paragraphs. State honestly where the LUT approach breaks down — e.g. it is a
small-signal method and does not predict large-signal settling; the interpolation error grows near the
moderate-inversion boundary; the flow assumes the topology is fixed and does not do topology selection.
Naming your own limits is a strength signal, not a weakness.\>

### 10.2 Scope limitation: this is a gain core, not a complete compressor

The envelope detector and the closed gain-control loop are outside the scope of this notebook. In a
complete dynamic-range compressor, $V_{\mathrm{ctrl}}$ would be driven by a rectifier and peak/RMS
detector whose time constants set the attack and release behaviour, and the loop stability of that
feedback path would require its own verification. **We therefore do not report a compression ratio,
threshold, knee, or attack/release characteristic, and no claim about compression behaviour should be
inferred from these results.** What is delivered and verified is the gain element itself, characterised
across its full control range — the block on which such a loop would be built.

### 10.3 Future work

1. Close the loop: envelope detector, attack/release time-constant design, loop stability across levels.
2. Extend the LUT flow to the detector's translinear devices, so the whole compressor is synthesised from
   one specification.
3. Silicon: submit through \<Tiny Tapeout / chipIgnite\> and correlate measurement against Table 7.


---
## 11. Conclusion

\<Five to eight sentences. Restate the contribution as a *method* with a *validated* outcome: an open,
cached, one-click-reproducible $g_m/I_D$ flow that sizes a subthreshold VGA core from specification to
DRC/LVS-clean layout, with post-layout FoM within \<x\>% of prediction, at \<y\> µA and \<z\> µV$_{\rm rms}$
input-referred noise. Close by pointing at what the artefact lets someone else do — re-synthesise the
block against their own specification by editing one dictionary. That sentence is the reason a
reproducibility-focused jury funds the trip.\>


---
## 12. References

[1] \<Prior-art hearing-aid / biomedical amplifier you target. Full IEEE-style citation with DOI.\>

[2] \<gm/ID methodology reference — e.g. Jespers & Murmann, *Systematic Design of Analog CMOS Circuits*.\>

[3] A. Adair, "Automated Current Mirror OTA Design and Optimization: From Specification to Layout,"
IEEE SSCS Code-a-Chip Travel Grant Award, ISSCC'25. \<repo link\>

[4] A. M. Elhelly, "Automated Design and Optimization of Current-Mode Logic Chains / Dividers,"
IEEE SSCS Code-a-Chip Travel Grant Award, ISSCC'26. \<repo link\>

[5] SkyWater Technology, *SKY130 Open Source PDK*. \<link\>

[6] \<pymoo, ngspice, Magic, Netgen, KLayout — cite the tools you depend on.\>

---

### Acknowledgements

\<Advisor, lab, any mentor. One or two lines.\>
